<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-09-multimodal-and-pretrained/lesson-9.6-multimodal-rag/notebooks/GCP_Capstone_9.6_MultimodalRAG.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 9.6 Multimodal RAG — Figure, Table and Segment Citations Through the One Contract, on the Lane
**Netsetos GenAI Engineering — GCP Capstone** · Module 9 · rebuilt on the live lane, 9 September 2026

The lesson the module has been walking towards, on live citations. The contract's four optional fields, proved on the kit's own class and on what `retrieve()` returns for a figure question today; the caption as the quote, the worker's beside a fresh one; a page crop pushed through the upload door and back as a figure citation; RRF inside `retrieve()`, and why on the lane it is fusion by construction; the model shown the figure it cites through `/v1/query`; DLP on the pixels through the kit's own `pii.py`; the budget and the admin row; and the finale - one figure question on three surfaces, the API, the MCP server and the A2A peer, with one chunk id. None of the three changed for Module 9. Media is a document.


## Setup
One generation client on global; the MCP server's and the A2A peer's deterministic URLs for the last cell.


In [ ]:
!pip install -q google-genai==2.22.0 google-cloud-storage==3.13.1 google-cloud-dlp==3.39.0 pypdfium2==5.13.0 Pillow==12.3.0 fastmcp==3.4.7 httpx==0.28.1 google-auth==2.57.1 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the kit: deploy/shared is the tool layer every lesson on the lane imports
BRANCH     = "main"        # the learner repo's branch: the notebooks and the kit (deploy/) ship there together

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google import genai
from google.genai import types

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
API_URL       = f"https://documind-api-{NUMBER}.{REGION}.run.app"
UPLOAD_BUCKET = f"{PROJECT_ID}-uploads"     # storage.tf: the bucket eventarc.tf watches - the corpus, media included
MEDIA_BUCKET  = f"{PROJECT_ID}-media"       # storage.tf: generated assets, 30-day lifecycle (a cache, not a record)
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": API_URL,
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MEMBER_SA   = os.environ["DOCUMIND_IMPERSONATE_SA"]
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"   # IAM admits it, no roster does (4.8, 7.2)

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - the contract gate fails a paste.
gen = genai.Client(enterprise=True, project=PROJECT_ID, location="global")   # every generate_content in this lesson
MCP_URL   = f"https://documind-mcp-{NUMBER}.{REGION}.run.app"      # 7.2's server
AGENT_URL = f"https://documind-agent-{NUMBER}.{REGION}.run.app"    # 8.4's A2A peer

print("kit:", KIT, "| API:", API_URL, "| media:", f"gs://{MEDIA_BUCKET}")


## Cell 1: The corpus's media, and the identities


In [ ]:
import json, requests, time
from google.cloud import storage

# THE ROUTES 9.4's Studio stands on, called the way the UI calls them: one ID token per request,
# minted AS the roster member, audience = the API (7.3's hour-long fuse never arms). The body names
# the tenant; the API checks the caller's email on that tenant's roster before it spends a paisa.
def api(path: str, body: dict | None = None, timeout: int = 120) -> tuple[int, dict | str]:
    """POST one API route as documind-ui-sa. Returns (status, json-or-text) - never raises on 4xx,
    because a refusal is data this module reads (the outsider cells)."""
    r = requests.post(f"{API_URL}{path}", json=body,
                      headers={"Authorization": f"Bearer {documind_tools._id_token(API_URL)}"}, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

# The corpus's media objects, read as YOU (the Colab credential is a project owner; the lane's
# services read them as their own accounts). One client, both buckets.
gcs = storage.Client(project=PROJECT_ID)

def media_objects(prefix: str = f"{TENANT}/") -> list[str]:
    """Every image, video or recording under the tenant's prefix of the uploads bucket."""
    return sorted(b.name for b in gcs.list_blobs(UPLOAD_BUCKET, prefix=prefix)
                  if b.name.lower().endswith((".png", ".jpg", ".jpeg", ".mp4", ".mp3")))

def gcs_bytes(uri: str) -> bytes:
    bucket, _, name = uri.removeprefix("gs://").partition("/")
    return gcs.bucket(bucket).blob(name).download_as_bytes()

FIG3  = f"gs://{UPLOAD_BUCKET}/{TENANT}/annual_report_2026_fig3.png"
INV   = f"gs://{UPLOAD_BUCKET}/{TENANT}/inv_2026_0412.png"
PAGE  = f"gs://{UPLOAD_BUCKET}/{TENANT}/payment_of_bonus_act_1965_p30.png"
VIDEO = f"gs://{UPLOAD_BUCKET}/{TENANT}/townhall_2026_q1.mp4"
POSH  = f"gs://{UPLOAD_BUCKET}/{TENANT}/posh_act_2013.pdf"

have = media_objects()
HAS_VIDEO = f"{TENANT}/townhall_2026_q1.mp4" in have
print("media in the corpus:", have or "NONE - run `make media` and `make ingest-corpus` from deploy/ (README: media is a document)")
print("video:", "present" if HAS_VIDEO else "absent (make media MEDIA_ARGS=--video, or drop a recording in) - the video cells will say so")
assert have, "no media under the tenant's prefix: nothing in this lesson can cite a figure until the corpus holds one"


In [ ]:
from google.auth import impersonated_credentials
from google.auth.transport.requests import Request

def id_token_as(service_account: str, audience: str) -> str:
    """A Google ID token minted AS a service account, for one audience, with the email (4.8, 7.3)."""
    source, _ = google.auth.default()
    target = impersonated_credentials.Credentials(source_credentials=source, target_principal=service_account,
                                                  target_scopes=["https://www.googleapis.com/auth/cloud-platform"])
    idc = impersonated_credentials.IDTokenCredentials(target, target_audience=audience, include_email=True)
    idc.refresh(Request())
    return idc.token

def api_as(service_account: str, path: str, body: dict) -> tuple[int, dict | str]:
    """The same route, as a DIFFERENT account - the outsider cells."""
    r = requests.post(f"{API_URL}{path}", json=body,
                      headers={"Authorization": f"Bearer {id_token_as(service_account, API_URL)}"}, timeout=120)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]


## Cell 2: The contract, on live citations
Four optional fields with defaults, one vocabulary, and a figure question whose citations come back in two kinds.


In [ ]:
import typing
from shared.documind_schemas import Citation
from shared.documind_tools import CITATION_KEYS

# THE CONTRACT, ON LIVE CITATIONS. Every brain in this course reads one shape, and two places build it:
# shared/documind_tools.py (CITATION_KEYS) and services/chat/tools.py. Module 9 added four OPTIONAL
# fields with defaults - kind, media_url, start, end - so a citation written before Module 9 comes through
# byte-identical, and a figure or a segment comes through with its locator. The vocabulary is the
# contract's, the indexer's and this notebook's: one, never a second.
KINDS = ('text', 'figure', 'table', 'segment')
assert set(typing.get_args(Citation.model_fields["kind"].annotation)) == set(KINDS)
assert CITATION_KEYS[-4:] == ("kind", "media_url", "start", "end")
print("Citation:", list(Citation.model_fields), "| kinds:", KINDS)

Q = "In Figure 3 of the annual report, which region's revenue declined between FY2025 and FY2026, and by how much?"
hits = documind_tools.retrieve(Q, tenant_id=TENANT, top_k=8, brain="direct")
by_kind = {}
for c in hits.get("citations", []):
    by_kind.setdefault(c.get("kind", "text"), []).append(c)
for kind, cs in by_kind.items():
    print(f"  {kind:8} x{len(cs)}   e.g. {cs[0]['source_uri'].rsplit('/', 1)[-1]:40} media_url={'yes' if cs[0].get('media_url') else '-'}")
assert "figure" in by_kind and "text" in by_kind, f"kinds seen: {sorted(by_kind)} - is the media ingested? (make media, make ingest-corpus)"
print("\nanswer:", (hits.get("answer") or "")[:220])
assert "EMEA" in (hits.get("answer") or ""), "the lane did not name EMEA"
FIG = by_kind["figure"][0]


## Cell 3: Prove backward compatibility, do not claim it
On the kit's own `Citation`: the old five-field dict, the old consumer that silently drops the new fields, the two failures the extension makes impossible.


In [ ]:
from pydantic import ValidationError

# PROVE BACKWARD COMPATIBILITY, DO NOT CLAIM IT - on the kit's own class, not a copy. A citation written
# before Module 9 carries five fields; through Citation it comes out with kind="text" and nothing else new.
old = {"chunk_id": "acme:hr_policy_2026#NP-03", "source_uri": "gs://b/hr_policy_2026.md",
       "page": None, "quote": "A confirmed E3 serves a notice period of 60 days.", "score": 0.94}
c = Citation(**old)
assert c.kind == "text" and c.media_url is None and c.start is None and c.end is None
print("pre-Module-9 citation ->", c.kind, "| unchanged fields:", all(getattr(c, k) == v for k, v in old.items()))

# The OLD consumer - the five-field projection that shipped before today - survives every live citation
# and DROPS the four new fields: not a crash, a figure rendered as plain text with no thumbnail. That is
# the silent failure, and why both projections in the kit were widened (CITATION_KEYS, chat/tools.py).
old_consumer = lambda c: {k: c[k] for k in ("chunk_id", "source_uri", "page", "quote", "score")}
projected = old_consumer(FIG)
print("old consumer on a live figure ->", sorted(projected), "(kind and media_url gone, nothing raised)")
assert "kind" not in projected

# The failures the extension makes impossible: a kind outside the vocabulary, and a media citation with no locator.
try:
    Citation(**{**old, "kind": "diagram"})
    raise SystemExit("an unknown kind was accepted")
except ValidationError as e:
    print("refused:", str(e).splitlines()[0][:80])

def usable(c: dict) -> bool:
    if c.get("kind") in ("figure", "table"):
        return bool(c.get("media_url"))
    if c.get("kind") == "segment":
        return bool(c.get("media_url")) and c.get("start") is not None
    return True

assert all(usable(c) for cs in by_kind.values() for c in cs), "a live media citation without a locator"
assert not usable({"kind": "segment", "quote": "someone said something"})
print("every live citation is usable; a segment without a start is caught at write time")


## Cell 4: The caption is the quote
The worker's prompt, run again on the same figure. The worker's caption is the one the retriever matched.


In [ ]:
# THE CAPTION IS THE QUOTE. A figure cannot be retrieved as pixels: retrieve() returns a quote, and a quote
# is text. The worker wrote this figure's caption at ingest with the prompt below (services/ingest/main.py,
# _describe_media) and embedded it; the image itself rides beside it as media_url. Run the same prompt
# here and read both - the worker's is the one the retriever matched.
PROMPT = ("Describe this figure for retrieval: one caption sentence, then the key facts it shows, then any table it "
          "contains as Markdown.")
fresh = gen.models.generate_content(model="gemini-3.6-flash",
    contents=[types.Part.from_uri(file_uri=FIG["media_url"], mime_type="image/png"), PROMPT],
    config=types.GenerateContentConfig(thinking_config=types.ThinkingConfig(thinking_level="LOW"))).text
numbers = lambda s: {w for w in "".join(ch if ch.isalnum() else " " for ch in s).split() if w.isdigit()}
print("the worker's caption (indexed) :", FIG["quote"][:260], "...")
print("\na fresh caption, same prompt   :", fresh[:260], "...")
print("\nnumbers both carry:", sorted(numbers(FIG["quote"]) & numbers(fresh)))
assert {"91", "96"} <= numbers(FIG["quote"]) | numbers(fresh) and "EMEA" in FIG["quote"] + fresh


## Cell 5: A page crop, through the door
pypdfium2 renders the Fourth Schedule's table; the signed PUT makes it an ingest; the lane cites it as a figure with the table as Markdown in the caption.


In [ ]:
import pypdfium2 as pdfium
from io import BytesIO
from PIL import Image
from IPython.display import display

# PAGE CROPS, THROUGH THE DOOR. Layout Parser (4.1, $10 per 1,000 pages) marks figure and table blocks on a
# page, which is what tells you where to crop at scale. Here the box is known - page 30 of the Payment of
# Bonus Act, the Fourth Schedule's table - and pypdfium2 renders it (the same job as PyMuPDF, without the
# AGPL question the exercise grid asks). The crop goes through 9.4's signed PUT, so it is an INGEST: the
# worker captions it with the table as Markdown, scans its pixels, and the lane cites it.
pdf = pdfium.PdfDocument(gcs_bytes(f"gs://{UPLOAD_BUCKET}/{TENANT}/payment_of_bonus_act_1965.pdf"))
page = pdf[29].render(scale=2.0).to_pil()
w, h = page.size
crop = page.crop((int(w * 0.10), int(h * 0.19), int(w * 0.92), int(h * 0.88)))
crop.thumbnail((1600, 1600))          # the caps: a 300-dpi crop of a full A4 page is ~8 MP, one per figure per document per tenant
buf = BytesIO(); crop.save(buf, "PNG", optimize=True); png = buf.getvalue()
print("crop:", crop.size, f"{len(png) // 1024} KB")
display(crop)

NAME = "payment_of_bonus_act_1965_p30_table.png"
status, door = api(f"/v1/media/upload-url?filename={NAME}&content_type=image/png&tenant_id={TENANT}")
assert status == 200, (status, door)
put = requests.put(door["url"], data=png, headers={"Content-Type": "image/png"}, timeout=60)
assert put.status_code in (200, 201), (put.status_code, put.text[:200])
got = None
for i in range(18):
    hits2 = documind_tools.retrieve("the set on and set off table of the Fourth Schedule, year by year", tenant_id=TENANT, top_k=8, brain="direct")
    got = next((c for c in hits2.get("citations", []) if c["source_uri"].endswith(NAME)), None)
    if got:
        break
    time.sleep(10)
assert got, "the crop never came back cited - read documind-ingest's log"
print(f"\nindexed after ~{(i + 1) * 10}s as kind={got['kind']} | caption: {got['quote'][:200]}")
assert got["kind"] == "figure" and "set" in got["quote"].lower()


## Cell 6: RRF fusion inside `retrieve()`
One entry point. On the lane the fusion is by construction - the caption is a chunk in the one index - and this cell shows what a second ranking would change: the order, never the contract.


In [ ]:
# RRF FUSION INSIDE retrieve(), NOT BESIDE IT. There is one retrieval entry point. Media candidates do not
# get their own function; they are fused into the same ranking, and the caller cannot tell there were two
# searches. On the lane that fusion is BY CONSTRUCTION: a figure's caption is a chunk in the one index, so
# the dense pass and 4.5's reranker rank it against the text chunks in a single list - no second index to
# fuse (9.5, D3). This cell shows what fusing a second ranking would do: split the live list by kind, fuse
# by rank, and compare with what the one retrieve() returned.
def rrf(rankings, k: int = 60):
    scores = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            scores[doc] = scores.get(doc, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda kv: -kv[1])

live = documind_tools.retrieve(Q, tenant_id=TENANT, top_k=8, brain="direct")["citations"]
ids = [c["chunk_id"] for c in live]
text_rank = [c["chunk_id"] for c in live if c.get("kind") == "text"]
media_rank = [c["chunk_id"] for c in live if c.get("kind") != "text"]
fused = [d for d, _ in rrf([text_rank, media_rank])]
print("live ranking, kinds :", [c.get("kind", "text") for c in live])
print("live ranking        :", [i.rsplit("#", 1)[-1] for i in ids][:6])
print("fused text+media    :", [i.rsplit("#", 1)[-1] for i in fused][:6])
assert set(fused) == set(ids), "fusion invented or lost a chunk"
print("\nthe live list already interleaves the kinds; a second index would change the ORDER, never the contract")


## Cell 7: Show the model the figure it cites
`/v1/query`: the generator attaches the figure as an image Part for every packed figure with a locator, and the UI shows it inline.


In [ ]:
# SHOW THE MODEL THE FIGURE IT CITES. A verbalised caption is enough to RETRIEVE a figure; it is not always
# enough to ANSWER from one - "which region fell" needs the chart, not the sentence about the chart. So the
# generator (services/rag-api/generator.py) appends an image Part for every packed figure that has a locator
# - only those: each crop is ~258+ tokens, and most questions are answered by the caption alone. The UI
# then shows [Fig n, p.N] inline through a signed URL (citations.py). This is /v1/query, the whole route.
status, body = api("/v1/query", {"query": Q, "tenant_id": TENANT, "top_k": 6, "stream": False, "brain": "direct"})
assert status == 200, (status, body)
print(body["answer"][:300])
for c in body["citations"]:
    print(f"  [{c['kind']:7}] {c['source_uri'].rsplit('/', 1)[-1]:40} p.{c.get('page')}  media_url={'yes' if c.get('media_url') else '-'}")
assert body["answerable"] and any(c["kind"] == "figure" for c in body["citations"]) and "EMEA" in body["answer"]


## Cell 8: DLP on the pixels
`shared/pii.py` grew `inspect_image`; the worker scans a figure's bytes before it indexes the caption. Here, the kit's own function on a PNG carrying the corpus's invented PAN.


In [ ]:
from shared import pii
from PIL import ImageDraw, ImageFont

# DLP ON THE PIXELS - by EXTENDING shared/pii.py, not beside it. Two DLP configurations that drift is the worst
# kind of compliance bug: the scan misses a type, the dashboard reports zero findings, and both look correct.
# inspect_image is in the kit now (same info-types, same likelihood floor, same no-quote rule), and the
# ingest worker scans a figure's BYTES before it indexes the caption (12.5). A page crop of a payslip is
# exactly as sensitive as the text of one, and the text scan that cleared the page never looked at the PNG.
# This cell draws a PNG carrying the corpus's INVENTED PAN and mobile (the invoice's) and scans it as you.
img = Image.new("RGB", (900, 260), "white")
d = ImageDraw.Draw(img)
font = ImageFont.load_default(size=36)
d.text((30, 50), "Employee payslip - PAN: AAAPZ1234C", fill="black", font=font)
d.text((30, 130), "Mobile: +919876543210", fill="black", font=font)
buf = BytesIO(); img.save(buf, "PNG"); png = buf.getvalue()

# WHERE THE PIXELS GO. Image inspection is offered in a short list of DLP locations - global, asia,
# asia-southeast1, europe, us and a few US regions - and NOT in asia-south1: the first figure the lane
# ingested came back "400 Image inspection is not supported in this location" and went to the DLQ. So
# the text scan stays in Mumbai (pii.LOCATION) and the pixels go to Singapore (pii.IMAGE_LOCATION), the
# same residency compromise 9.3's Chirp table draws, and a decision DLP_IMAGE_LOCATION can change.
print("text scan in", pii.LOCATION, "| image scan in", pii.IMAGE_LOCATION)
findings = pii.inspect_image(png, "image/png")
print("findings:", findings)
types_found = {f["info_type"] for f in findings}
assert types_found & {"INDIA_PAN_INDIVIDUAL", "PHONE_NUMBER"}, "DLP read nothing off the picture"
assert all("quote" not in f and "value" not in f for f in findings), "a findings record must never carry the value"
assert pii.inspect_image(b"not an image", "video/mp4") == []
print("types only, never the value: a findings record that quotes the PAN it found has moved the PAN into your audit store")


## Cell 9: Per-tenant media budget, in rupees


In [ ]:
# Per-tenant media budget, in rupees.
USD_INR = 85
MEDIA_PRICES = {          # verified 2026-09-05, and dated because they move
    'image_generate': 0.039,      # per image, gemini-3.1-flash-image
    'transcribe_min': 0.005,      # per audio-minute, gemini-3.5-transcribe
    'layout_page':    0.010,      # per page, Layout Parser ($10 / 1K pages)
    'embed_1k':       0.00013,    # per 1K tokens, gemini-embedding-2-preview
}


def media_spend_inr(images=0, audio_min=0, pages=0, embed_tokens=0) -> float:
    usd = (images * MEDIA_PRICES['image_generate']
           + audio_min * MEDIA_PRICES['transcribe_min']
           + pages * MEDIA_PRICES['layout_page']
           + (embed_tokens / 1000) * MEDIA_PRICES['embed_1k'])
    return round(usd * USD_INR, 2)


def budget_check(spend_inr: float, cap_inr: float) -> tuple[str, float]:
    """Returns (state, fraction). Alert BEFORE the cap, refuse AT it."""
    frac = spend_inr / cap_inr if cap_inr else 0.0
    if frac >= 1.0:
        return 'REFUSE', frac
    if frac >= 0.8:
        return 'ALERT', frac
    return 'ok', frac


CAP = 2000.0        # rupees per tenant per month - and see what that does
print(f'{"scenario":34} {"INR":>9}  {"state":>7}  {"of cap":>7}')
print('-' * 64)
for label, kw in (
        ('light: 20 figures, 30 min audio', dict(images=20, audio_min=30, pages=200)),
        ('steady: 140 figures, 3.3h audio', dict(images=140, audio_min=200, pages=1400)),
        ('normal: 200 figures, 5h audio',   dict(images=200, audio_min=300, pages=2000)),
        ('runaway: 2000 figures',           dict(images=2000, audio_min=600, pages=20000))):
    inr = media_spend_inr(**kw)
    state, frac = budget_check(inr, CAP)
    print(f'{label:34} {inr:>9,.0f}  {state:>7}  {frac:>6.0%}')

print()
print('Read the middle row again: ORDINARY usage refuses at a Rs 2,000 cap.')
print('That is not the tenant misbehaving - it is the cap being wrong. A number')
print('picked because it looked round, before anyone measured a month, only ever')
print('fails in one direction: it blocks real work and calls it governance.')
print('Measure first, THEN set the cap, and re-derive it when the corpus grows.')
print()
print('ALERT at 80% and REFUSE at 100% is the same two-layer shape as 10.3 and')
print('12.6: the alert is reactive and tells a human, the refusal is preventive')
print('and does not need one. Media makes the gap matter - one runaway ingest')
print('of a video library outspends a month of chat before anybody reads an email.')


## Cell 10: The admin row - media spend joins `tenant_daily`


In [ ]:
# The admin row: media spend joins tenant_daily.
SQL = '''-- Extends the tenant_daily view from 12.3. Media is a separate cost
-- CLASS, not a separate table: one place to look for "what did this tenant
-- cost me", or nobody looks in both.
SELECT
  tenant_id,
  DATE(ts) AS day,
  COUNTIF(action = 'media.generate')                       AS images_generated,
  COUNTIF(action = 'media.transcribe')                     AS audio_transcribed,
  ROUND(SUM(CAST(JSON_VALUE(meta, '$.cost_inr') AS FLOAT64)), 2) AS media_inr,
  ROUND(SUM(CAST(JSON_VALUE(meta, '$.cost_inr') AS FLOAT64)), 2)
    / NULLIF(ANY_VALUE(CAST(JSON_VALUE(meta, '$.cap_inr') AS FLOAT64)), 0) AS frac_of_cap
FROM `documind.audit_events`
WHERE action LIKE 'media.%'
  AND DATE(ts) >= DATE_SUB(CURRENT_DATE(), INTERVAL 30 DAY)
GROUP BY tenant_id, day
ORDER BY media_inr DESC
'''
with open('media_daily.sql', 'w') as f:
    f.write(SQL)
print('media_daily.sql written')
print()
print('It reads the AUDIT bucket, not a counter the services keep. The audit row')
print('was written at generation time by shared/audit_log.emit - so the bill and')
print('the compliance record come from the same fact, and cannot disagree.')


## Cell 11: Look how far
One figure question. The API, the MCP server through 7.3's credential path, the A2A peer through 8.4's `send()`. One chunk id.


In [ ]:
import uuid
import httpx
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport

# LOOK HOW FAR: ONE FIGURE QUESTION, THREE SURFACES, ONE CHUNK. The API (Module 4), the MCP server (7.2, a
# token per request as the roster member - 7.3's path), and the A2A peer (8.4, which knows DocuMind only
# through the MCP server, its own account on acme's roster). None of the three changed for Module 9: a
# figure is a document, and the contract carried kind and media_url through every one of them.
# 1. the API
status, via_api = api("/v1/query", {"query": Q, "tenant_id": TENANT, "top_k": 6, "stream": False, "brain": "direct"})
api_figs = {c["chunk_id"] for c in via_api["citations"] if c["kind"] == "figure"}

# 2. the MCP server: the same tool 7.3's agent and 8.1's mixed tool list call
async def mcp_retrieve(question: str) -> dict:
    headers = {"Authorization": f"Bearer {id_token_as(MEMBER_SA, MCP_URL)}"}
    async with Client(StreamableHttpTransport(f"{MCP_URL}/mcp", headers=headers)) as c:
        r = await c.call_tool("retrieve", {"query": question, "tenant": TENANT, "top_k": 6})
        out = r.data if getattr(r, "data", None) is not None else r
        return json.loads(out) if isinstance(out, str) else out

via_mcp = await mcp_retrieve(Q)
mcp_figs = {c["chunk_id"] for c in via_mcp.get("citations", []) if c.get("kind") == "figure"}

# 3. the A2A peer: 8.4's send(), a credential per request (httpx calls auth_flow every time)
class IdTokenAuth(httpx.Auth):
    def __init__(self, audience: str, service_account: str = MEMBER_SA):
        self.audience, self.service_account = audience, service_account
    def auth_flow(self, request):
        request.headers["Authorization"] = f"Bearer {id_token_as(self.service_account, self.audience)}"
        yield request

def send(base: str, text: str, client: httpx.Client) -> tuple:
    body = {"jsonrpc": "2.0", "id": str(uuid.uuid4()), "method": "message/send",
            "params": {"message": {"role": "user", "kind": "message", "messageId": str(uuid.uuid4()),
                                   "parts": [{"kind": "text", "text": text}]}}}
    j = client.post(f"{base}/", json=body, timeout=180).json()
    if "error" in j:
        return j, ""
    task = j["result"]
    texts = [pt.get("text", "") for a in task.get("artifacts", []) for pt in a.get("parts", []) if pt.get("kind") == "text"]
    return task, "\n".join(t for t in texts if t)

task, via_peer = send(AGENT_URL, Q, httpx.Client(auth=IdTokenAuth(AGENT_URL), timeout=180))

print("API  cited figure chunks:", sorted(api_figs))
print("MCP  cited figure chunks:", sorted(mcp_figs))
print("peer:", task.get("status", {}).get("state"), "|", via_peer[:200])
assert api_figs & mcp_figs, "the API and the MCP server cited different figure chunks"
assert "EMEA" in via_peer, "the peer did not relay the figure's answer: " + via_peer[:200]
print("\none figure, three surfaces, one chunk:", sorted(api_figs & mcp_figs)[0])


## Done - Module 9 is complete
- ✅ The contract's four optional fields, proved on the kit's class and on live citations
- ✅ The old consumer survives and silently drops; the two impossible failures refused
- ✅ The caption is the quote: the worker's beside a fresh one
- ✅ A page crop through the upload door, back as a figure citation with the table in its caption
- ✅ RRF inside `retrieve()`; on the lane, fusion by construction
- ✅ The model shown the figure it cites; the UI shows it inline
- ✅ DLP on the pixels through the kit's own `pii.py`; budget; the admin row
- ✅ One figure question, three surfaces, one chunk id - and none of the three changed

**Where this goes:** Module 10 fine-tunes on the corpus's own questions; Module 12 is the platform every service in this module runs on. The figure you cited today was drawn from the annual report's own table by `make media` - the report has referenced it since Module 4, and it has existed since this morning.
